# 1. Problem Definition
The business goal of this project is to predict final student outcomes based on demographic, social, and school-related features. To prevent trivial data leakage, we explicitly drop intermediate grades (G1 and G2), forcing the model to learn from behavior rather than past test scores. We tackle this using two approaches: a **regression** model to predict the exact final grade (`G3`, continuous from 0-20), and a **classification** model to predict if a student is at risk of failing (`pass`, binary where 1 means G3 ≥ 10), allowing schools to identify and assist struggling students early.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)

# Load both datasets from the .devcontainer folder
df_mat = pd.read_csv('.devcontainer/student-mat.csv', sep=';')
df_por = pd.read_csv('.devcontainer/student-por.csv', sep=';')

# Combine them to maximize our row count
df = pd.concat([df_mat, df_por], ignore_index=True)

# Create binary classification target (Pass = 1 if G3 >= 10, else 0)
df['pass'] = (df['G3'] >= 10).astype(int)

# Drop G1 and G2 to prevent data leakage
df = df.drop(columns=['G1', 'G2'])

df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G3,pass
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,no,4,3,4,1,1,3,6,6,0
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,no,5,3,3,1,1,3,4,6,0
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,no,4,3,2,2,3,3,10,10,1
3,GP,F,15,U,GT3,T,4,2,health,services,...,yes,3,2,2,1,1,5,2,15,1
4,GP,F,16,U,GT3,T,3,3,other,other,...,no,4,3,2,1,2,5,4,10,1
